In [ ]:
gen_report = False

In [ ]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"
df.loc[df["best_of_n"].isna(), "best_of_n"] = 1
df.loc[df["dataset_tt"].isna(), "dataset_tt"] = "one_shape"
df.loc[df["sampling_temperature_tt"].isna(), "sampling_temperature_tt"] = df["sampling_temperature"]

df.loc[(df["num_iterations"] == 0) | (df["best_of_n"] == 1), "test_time_mode"] = "-"

df.rename(columns={"test_time_training_accuracy": "test_time_mutual_accuracy"}, inplace=True)
df.rename(columns={"test_time_self_play_accuracy": "test_time_self_accuracy"}, inplace=True)

df["test_time_mode"] = df["test_time_mode"].replace("adaptation", "sample_adaptation")
df["dataset"] = df["dataset"].replace("shape", "shape1")
df["dataset_tt"] = df["dataset_tt"].replace("two_shape", "shape2")
df["dataset_tt"] = df["dataset_tt"].replace("shape_unique_double_attribute", "dual_attribute_shape")
# df["dataset_tt"] = df["dataset_tt"].replace("shape_unique_double_attribute", "single_attribute_shape")

df.loc[df["baseline"].isna(), "baseline"] = False

df.loc[df["baseline"], "test_time_mode"] = "-"


df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


In [ ]:
def filter_df(filters, df=df, sort_by=["message_length", "message_length_tt", "learning_rate_tt", "num_iterations"]):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=sort_by).reset_index(drop=True)

    

In [ ]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    # if 'mutual_play_accuracy' in df:
    #     max_col = 'mutual_play_accuracy'
    # else:
    #     max_col = 'test_accuracy'
    # max_val = pd.to_numeric(df[max_col]).max()

    # def highlight_max_row(row):
    #     if  pd.to_numeric(row[max_col]) == max_val:
    #         return ['font-weight: bold; background-color: #ffff99'] * len(row)
    #     else:
    #         return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            # .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(
                lambda x: (
                    f"{int(x)}"                      # 20.0 -> 20
                    if isinstance(x, float) and x.is_integer()
                    else f"{x:.0e}"                  # small numbers -> scientific
                    if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01
                    else x
                )
            )
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#111011",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [ ]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [ ]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["message_length", "message_length_tt", "seed"], max_col="test_time_mutual_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[pd.to_numeric(section[max_col]).idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols)


def mean_and_std(df, config_cols=["dataset", "message_length", "message_length_tt", "dataset_tt", "test_time_mode"], metrics=["test_time_self_accuracy", "test_time_mutual_accuracy"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]
        
        rows.append(
            {col: chunk[col].iloc[0] for col in config_cols} | 
            {metric: f"{pd.to_numeric(chunk[metric]).mean() * 100:.1f} ± {pd.to_numeric(chunk[metric]).std() * 100:.1f}" for metric in metrics}
        )

    out = pd.DataFrame(rows)
    return out.sort_values(by=config_cols)
    

In [ ]:
import matplotlib.pyplot as plt

def plot(dfs, labels=("Baseline", "VQEL", "VQEL + TTA (Batch)", "scaling", "VQEL + TTA (Dataset)", "VQEL + TTA (Sample)"), name="name"):
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

    all_x = []

    for df, label in zip(dfs, labels):
        df = df.copy()

        # Split "mean ± std"
        df[["mean_acc", "std_acc"]] = (
            df["test_time_mutual_accuracy"]
            .str.split("±", expand=True)
            .astype(float)
        )

        df = df.sort_values("message_length_tt")

        x = df["message_length_tt"]
        y = df["mean_acc"]
        err = df["std_acc"]

        all_x.extend(x.tolist())

        ax.plot(x, y, marker="o", linewidth=2, label=label)
        # ax.plot(x, y, linewidth=2, label=label)

        ax.fill_between(x, y - err, y + err, alpha=0.2)

    ax.set_xlabel("Message Length", fontsize=12)
    ax.set_ylabel("Accuracy under Distribution Shift (\%)", fontsize=12)

    ax.set_xticks(sorted(set(all_x)))
    ax.tick_params(axis="both", labelsize=10)

    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    
    plt.savefig(
        f"assets/{name}.pdf",
        format="pdf",
        bbox_inches="tight"
    )

    plt.show()


In [ ]:
clear()

---

In [ ]:
backbone_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "learning_rate_phase1", 
    "learning_rate_phase2_a", 
    "learning_rate_phase2_b",
    "test_time_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "path",
]

backbone_cols_report = [
    "seed",
    "message_length",
    "agent_a_training_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

baseline_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

baseline_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

scaling_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

scaling_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

adapt_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

adapt_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

In [ ]:
write(
"""
- <b>Shape1</b>: Contains only one shape.

- <b>Shape2</b>: Contains exactly two shapes.

- <b>Shape12</b>: Contains one or two shapes.

- <b>MNIST1</b>: Contains one digit on either the left or right side of the image (the other side is empty).

- <b>MNIST2</b>: Contains two digits.

- <b>ImageNet_same_class</b>: Distractors and targets belong to the same class.

- <b>Single Attribute Shape</b>: Candidates in a batch have two attributes; the value of one attribute is different for all images,
while there is no restriction on the value of the other attribute—it may be the same for some images and different for others.

- <b>Dual-attribute shape</b>: Candidates in a batch have two attributes, and all candidates can be distinguished from one another 
if and only if the values of both attributes are described.
"""
)

# Shape1

## Baseline

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "dialogued_checkpoint": "None",
    "baseline": True,
}, sort_by=["seed"])

res[backbone_cols]

In [ ]:
mean = mean_and_std(res, config_cols = ["dataset", "message_length"], metrics = ["mutual_play_accuracy"])
to_html(mean)
mean

In [ ]:
res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "dialogued_checkpoint": "!None",
    "baseline": True,
}, sort_by=["message_length_tt", "seed"])
# res[baseline_cols]

In [ ]:
baseline = mean_and_std(res)
# baseline

## Backbone

In [ ]:
add_heading(2, "Shape1")
add_heading(3, "Base Model")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "baseline": False,
    "message_length": "[3, 4]",
    "test_time_mode": "-",
    "dialogued_checkpoint": "None",
}, sort_by=["message_length"])
res = extract_maxes(res, max_col="mutual_play_accuracy")

res[backbone_cols]

In [ ]:
mean = mean_and_std(res, config_cols = ["dataset", "message_length"], metrics = ["self_play_accuracy_a", "mutual_play_accuracy"])
to_html(mean)
mean

## VQ-NoTT

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "dataset_tt": "shape2",
    "baseline": False,
    "message_length": "[3, 4]",
    "test_time_mode": "-"
}, sort_by=["message_length_tt", "seed"])

to_html(res[baseline_cols_report])

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
# VQ_NoTT

## Scaling

In [ ]:
add_heading(3, "Scaling")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "scaling"
}, sort_by=["message_length", "message_length_tt", "seed", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [ ]:
final = extract_maxes(res)
# final[scaling_cols]

In [ ]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [ ]:
add_heading(3, "Adaptation")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": ["batch_adaptation"],
}, sort_by=["message_length_tt","seed", "learning_rate_tt"])

to_html(res[adapt_cols_report])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(final)
# batch_adapt

## Dataset Adaptation

In [ ]:
res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": ["dataset_adaptation"],
}, sort_by=["message_length_tt","seed", "learning_rate_tt"])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(final)
# dataset_adapt

In [ ]:
plot([baseline, VQ_NoTT, batch_adapt, scaling, dataset_adapt], name="shape")

# Shape12

## Baseline

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape12",
    "baseline": True,
    "dialogued_checkpoint": "None"
}, sort_by=["seed"])

res[backbone_cols]

In [ ]:
mean = mean_and_std(res, config_cols = ["dataset", "message_length"], metrics = ["mutual_play_accuracy"])
to_html(mean)
mean

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape12",
    "baseline": True,
    "dialogued_checkpoint": "!None"
}, sort_by=["message_length_tt", "seed"])

# res[baseline_cols]

In [ ]:
baseline = mean_and_std(res)
# baseline

## Backbone

In [ ]:
add_heading(2, "Shape12")
add_heading(3, "Base Model")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "baseline": False,
    "dialogued_checkpoint": "None",
}, sort_by=["seed", "message_length"])

to_html(res[backbone_cols_report])

# res[backbone_cols]

In [ ]:
mean = mean_and_std(res, config_cols = ["dataset", "message_length"], metrics = ["self_play_accuracy_a", "mutual_play_accuracy"])
mean

## VQ-NoTT

In [ ]:
res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "dataset_tt": "shape3",
    "baseline": False,
    "message_length": "[3, 4]",
    "test_time_mode": "-"
}, sort_by=["message_length_tt", "seed"])

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
# VQ_NoTT

## Scaling

In [ ]:
add_heading(3, "Scaling")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "scaling"
}, sort_by=["message_length", "message_length_tt", "sampling_temperature_tt", "best_of_n"])

# res[scaling_cols]

## Adaptation

In [ ]:
add_heading(3, "Adaptation")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": ["batch_adaptation"],
}, sort_by=["test_time_mode", "message_length_tt", "seed", "learning_rate_tt", "num_iterations"])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
adapt = mean_and_std(final)
# adapt

In [ ]:
# plot([baseline, VQ_NoTT], adapt)

# MNIST

## Baseline

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "mnist1",
    "baseline": True,
    "dialogued_checkpoint": "None"
}, sort_by=["seed"])

res[backbone_cols]

In [ ]:
mean_and_std(res)

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "mnist1",
    "baseline": True,
    "dialogued_checkpoint": "!None"
}, sort_by=["message_length_tt", "seed"])

# res[baseline_cols]

In [ ]:
baseline = mean_and_std(res)
# baseline

## Backbone

In [ ]:
add_heading(2, "MNIST")
add_heading(3, 'Base Model')
res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "baseline": False,
    "test_time_mode": "-",
    "dataset_tt": "mnist1",
    "message_length": "[3, 4]",
}, sort_by=["message_length", "message_length_tt"])


res[backbone_cols]

In [ ]:
mean = mean_and_std(res, config_cols = ["dataset", "message_length"], metrics = ["self_play_accuracy_a", "mutual_play_accuracy"])
mean

## VQ-NoTT

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "dataset_tt": "mnist2",
    "baseline": False,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt", "seed"])

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
# VQ_NoTT

## Scaling

In [ ]:
add_heading(3, "Scaling")
res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "baseline": False,
    "test_time_mode": "scaling",
    "sampling_temperature_tt": "1e-02"
}, sort_by=["message_length", "message_length_tt", "sampling_temperature_tt", "best_of_n"])

# res[scaling_cols]

In [ ]:
final = extract_maxes(res)
# final[scaling_cols]

In [ ]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [ ]:
add_heading(3, "Adaptation")

res = filter_df({
    "dataset": "mnist1",
    "VQEL": True,
    "test_time_mode": ["batch_adaptation"],
}, sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(final)
# batch_adapt

## Dataset Adaptation

In [ ]:
add_heading(3, "Adaptation")

res = filter_df({
    "dataset": "mnist1",
    "VQEL": True,
    "test_time_mode": ["dataset_adaptation"],
}, sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(final)
# dataset_adapt

In [ ]:
plot([baseline, VQ_NoTT, batch_adapt, scaling, dataset_adapt], name="mnist")

# ImageNet

## Baseline

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "imagenet",
    "baseline": True,
    "message_length": "[2, 3, 4]",
    "dialogued_checkpoint": "None",
}, sort_by=["message_length", "seed"])

res[backbone_cols]

In [ ]:
mean_and_std(res)

In [ ]:
res = filter_df({
    "dataset": "imagenet",
    "baseline": True,
    "message_length": "[2, 3, 4]",
    "dialogued_checkpoint": "!None",
}, sort_by=["message_length_tt", "seed"])
# res[baseline_cols]

In [ ]:
baseline = mean_and_std(res)
# baseline

## Backbone

In [ ]:
add_heading(2, "ImageNet")
add_heading(3, "Base Model")

res = filter_df({
    "dataset": "imagenet",
    "sim": "cosine",
    "baseline": False,
    "test_time_mode": "-",
    "dataset_tt": "imagenet",
    "message_length": "[2, 3, 4]"
}, sort_by=["message_length","seed", "message_length_tt"])


res[backbone_cols]

In [ ]:
mean = mean_and_std(res, config_cols = ["dataset", "message_length"], metrics = ["self_play_accuracy_a", "mutual_play_accuracy"])
mean

## VQ-NoTT

In [ ]:
add_heading(3, "Baseline")
res = filter_df({
    "dataset": "imagenet",
    "dataset_tt": "imagenet_same_class",
    "baseline": False,
    "test_time_mode": "-",
    "message_length": "[2, 3, 4]"
}, sort_by=["message_length", "message_length_tt", "seed"])

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
# VQ_NoTT

## Scaling

In [ ]:
add_heading(3, "Scaling")
res = filter_df({
    "dataset": "imagenet",
    "baseline": False,
    "test_time_mode": 'scaling'
}, sort_by=["message_length_tt", "sampling_temperature_tt", "best_of_n"])

# res[scaling_cols]

In [ ]:
scaling = mean_and_std(res)
# scaling

## Batch Adaptation

In [ ]:
add_heading(3, "Adaptation")
res = filter_df({
    "dataset": "imagenet",
    "baseline": False,
    "test_time_mode": ["batch_adaptation"],
    "message_length": "[2, 3, 4]",
}, sort_by=["test_time_mode", "message_length_tt", "seed", "learning_rate_tt", "num_iterations"])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(final)
# batch_adapt

## Dataset Adaptation

In [ ]:
add_heading(3, "Adaptation")
res = filter_df({
    "dataset": "imagenet",
    "baseline": False,
    "test_time_mode": ["dataset_adaptation"],
    "message_length": "[2, 3, 4]",
}, sort_by=["test_time_mode", "message_length_tt", "seed", "learning_rate_tt", "num_iterations"])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(final)
# dataset_adapt

In [ ]:
plot([baseline, VQ_NoTT, batch_adapt, scaling, dataset_adapt], name="imagenet")

# Shape Single Attribute


## Baseline

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "sim": "cosine",
    "baseline": True,
    "dialogued_checkpoint": "None"
}, sort_by=["seed"])

res[backbone_cols]

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "sim": "cosine",
    "baseline": True,
    "dialogued_checkpoint": "!None"
}, sort_by=["message_length_tt", "seed"])

# res[baseline_cols]

In [ ]:
baseline = mean_and_std(res)
# baseline

## Backbone

In [ ]:
add_heading(2, "Single Attribute  Shape")
add_heading(3, "Base Model")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "sim": "cosine",
    "baseline": False,
    "message_length": "[3, 4]",
    "test_time_mode": "-",
    "pretrained_checkpoint_a": "None"
}, sort_by=["seed", "message_length", "message_length_tt"])

to_html(res[backbone_cols_report])

res[backbone_cols]

In [ ]:
mean = mean_and_std(res, config_cols = ["dataset", "message_length"], metrics = ["self_play_accuracy_a", "mutual_play_accuracy"])
mean

## VQ-NoTT

In [ ]:
add_heading(3, "Baseline")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "baseline": False,
    "test_time_mode": "-",
    "message_length": "[3, 4]",
    "pretrained_checkpoint_a": "!None"
}, sort_by=["message_length", "message_length_tt", "seed"])

to_html(res[baseline_cols_report])

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
# VQ_NoTT

## Scaling

In [ ]:
add_heading(3, "Scaling")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "baseline": False,
    "message_length": "[3, 4]",
    "test_time_mode": 'scaling'
}, sort_by=["message_length_tt", "sampling_temperature_tt", "best_of_n"])

# res[scaling_cols]

## Adaptation

In [ ]:
add_heading(3, "Adaptation")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "baseline": False,
    "message_length": "[3, 4]",
    "test_time_mode": ["batch_adaptation", "dataset_adaptation"],
}, sort_by=["message_length", "test_time_mode", "message_length_tt", "seed", "learning_rate_tt", "num_iterations"])

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
adapt = mean_and_std(final)
# adapt

In [ ]:
# plot([baseline, VQ_NoTT, adapt])